# Homework 2 - F1 Analysis with PySpark

This notebook answers all six README questions using the same Databricks volume path and PySpark workflow shown in `etl_pipeline_data_quality.ipynb`.

## Read And Prepare The Data

I follow the ETL notebook's pattern by reading the raw CSV files directly from `/Volumes/gr5069/raw/f1_data/`. I then cast the key columns into numeric or date types before answering the homework questions so that aggregations, joins, and window functions behave correctly.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
data_path = "/Volumes/gr5069/raw/f1_data"

pit_stops_raw = spark.read.csv(f"{data_path}/pit_stops.csv", header=True)
drivers_raw = spark.read.csv(f"{data_path}/drivers.csv", header=True)
races_raw = spark.read.csv(f"{data_path}/races.csv", header=True)
results_raw = spark.read.csv(f"{data_path}/results.csv", header=True)

In [0]:
pit_stops = (
    pit_stops_raw
    .select(
        F.col("raceId").cast("int").alias("raceId"),
        F.col("driverId").cast("int").alias("driverId"),
        F.col("stop").cast("int").alias("stop_number"),
        F.col("lap").cast("int").alias("lap"),
        F.col("time").alias("pit_time_of_day"),
        F.col("duration").cast("double").alias("pit_duration_seconds"),
        F.col("milliseconds").cast("int").alias("pit_milliseconds")
    )
    .filter(F.col("pit_milliseconds").isNotNull())
)

drivers = (
    drivers_raw
    .select(
        F.col("driverId").cast("int").alias("driverId"),
        "driverRef",
        F.col("number").cast("int").alias("number"),
        "code",
        "forename",
        "surname",
        F.to_date("dob").alias("dob"),
        "nationality",
        "url"
    )
)

races = (
    races_raw
    .select(
        F.col("raceId").cast("int").alias("raceId"),
        F.col("year").cast("int").alias("year"),
        F.col("round").cast("int").alias("round"),
        F.col("circuitId").cast("int").alias("circuitId"),
        F.col("name").alias("race_name"),
        F.to_date("date").alias("race_date"),
        "time"
    )
)

results = (
    results_raw
    .select(
        F.col("resultId").cast("int").alias("resultId"),
        F.col("raceId").cast("int").alias("raceId"),
        F.col("driverId").cast("int").alias("driverId"),
        F.col("constructorId").cast("int").alias("constructorId"),
        F.col("grid").cast("int").alias("grid_position"),
        F.col("position").cast("int").alias("position"),
        F.col("positionOrder").cast("int").alias("finish_position"),
        F.col("points").cast("double").alias("points"),
        F.col("laps").cast("int").alias("laps"),
        F.col("milliseconds").cast("int").alias("race_milliseconds"),
        F.col("statusId").cast("int").alias("statusId")
    )
)

## Question 1

To find the average time each driver spent in the pit stop for each race, I aggregate the `pit_stops` table at the `(raceId, driverId)` level and average `milliseconds`. To add the slowest and fastest pit stop for each race, I separately aggregate the same table at the `raceId` level with `min` and `max`, then join those race-level values back onto the driver-level summary together with race and driver labels.

In [0]:
race_stop_extremes = (
    pit_stops
    .groupBy("raceId")
    .agg(
        F.min("pit_milliseconds").alias("race_fastest_pit_stop_ms"),
        F.max("pit_milliseconds").alias("race_slowest_pit_stop_ms")
    )
)

driver_race_pit_summary = (
    pit_stops
    .groupBy("raceId", "driverId")
    .agg(
        F.avg("pit_milliseconds").alias("avg_pit_stop_ms"),
        F.count("*").alias("number_of_stops")
    )
    .join(race_stop_extremes, on="raceId", how="left")
    .join(races.select("raceId", "year", "round", "race_name", "race_date"), on="raceId", how="left")
    .join(drivers.select("driverId", "code", "forename", "surname"), on="driverId", how="left")
    .withColumn("avg_pit_stop_seconds", F.round(F.col("avg_pit_stop_ms") / 1000, 3))
    .withColumn("race_fastest_pit_stop_seconds", F.round(F.col("race_fastest_pit_stop_ms") / 1000, 3))
    .withColumn("race_slowest_pit_stop_seconds", F.round(F.col("race_slowest_pit_stop_ms") / 1000, 3))
)

display(
    driver_race_pit_summary
    .select(
        "year",
        "round",
        "race_name",
        "race_date",
        "driverId",
        "code",
        "forename",
        "surname",
        "number_of_stops",
        "avg_pit_stop_seconds",
        "race_fastest_pit_stop_seconds",
        "race_slowest_pit_stop_seconds"
    )
    .orderBy("year", "round", "avg_pit_stop_seconds", "driverId")
)

The `groupBy("raceId", "driverId")` and `avg("pit_milliseconds")` commands compute each driver's average pit-stop time within a race, while `count("*")` shows how many recorded stops contributed to that average. I then use a second aggregation with `min` and `max` to capture the fastest and slowest stop in each race, `join(...)` those race-level values back to the driver-level summary, and `withColumn(...)` plus `round(...)` to convert milliseconds into a more readable seconds format. An alternative route would be to register temp views and solve the same problem with Spark SQL aggregates.

## Question 2

I reuse the average pit-stop result from Question 1, then join it to the `results` dataset so each driver-race row also has a finishing position. Once the pit-stop averages and finishing positions live on the same row, I sort the output within each race by finishing position to rank-order the pit-stop averages by race result.

In [0]:
pit_time_by_finish = (
    results
    .filter(F.col("finish_position").isNotNull())
    .select("raceId", "driverId", "finish_position")
    .join(
        driver_race_pit_summary.select("raceId", "driverId", "avg_pit_stop_ms", "avg_pit_stop_seconds"),
        on=["raceId", "driverId"],
        how="left"
    )
    .join(races.select("raceId", "year", "round", "race_name"), on="raceId", how="left")
    .join(drivers.select("driverId", "code", "forename", "surname"), on="driverId", how="left")
    .withColumn(
        "finish_rank",
        F.dense_rank().over(Window.partitionBy("raceId").orderBy(F.col("finish_position").asc()))
    )
)

display(
    pit_time_by_finish
    .select(
        "year",
        "round",
        "race_name",
        "finish_position",
        "finish_rank",
        "driverId",
        "code",
        "forename",
        "surname",
        "avg_pit_stop_seconds"
    )
    .orderBy("year", "round", "finish_position", "driverId")
)

The `join(...)` between `results` and `driver_race_pit_summary` attaches each driver's average pit-stop time to the row that contains that driver's finishing position in the same race. I use `dense_rank().over(Window.partitionBy("raceId").orderBy("finish_position"))` to make the race-level ranking explicit, and the final `orderBy(...)` returns the table in finish order so the pit-stop averages are presented in the same order as the race result. The same analysis could also be written as a SQL window query over temporary views.

## Question 3

To fill missing driver codes, I use a deterministic rule based only on the `drivers` dataset: if `code` is null, blank, or the placeholder string `\\\\N`, I build a replacement from the first three alphabetic characters of the surname in uppercase, which matches the homework example `ALO` for Alonso. For very short surnames, I fall back to the cleaned `driverRef` and pad the result to three characters so every driver ends with a three-letter code.

In [0]:
drivers_completed = (
    drivers
    .withColumnRenamed("code", "original_code")
    .withColumn(
        "original_code",
        F.when(
            F.col("original_code").isNull()
            | (F.trim(F.col("original_code")) == "")
            | (F.trim(F.col("original_code")) == "\\N"),
            F.lit(None)
        ).otherwise(F.col("original_code"))
    )
    .withColumn("clean_surname", F.upper(F.regexp_replace(F.col("surname"), "[^A-Za-z]", "")))
    .withColumn("clean_driver_ref", F.upper(F.regexp_replace(F.col("driverRef"), "[^A-Za-z]", "")))
    .withColumn(
        "code",
        F.when(
            F.col("original_code").isNull(),
            F.when(F.length(F.col("clean_surname")) >= 3, F.substring(F.col("clean_surname"), 1, 3))
             .otherwise(F.rpad(F.substring(F.col("clean_driver_ref"), 1, 3), 3, "X"))
        ).otherwise(F.col("original_code"))
    )
    .drop("clean_surname", "clean_driver_ref")
)

display(
    drivers_completed
    .filter(F.col("original_code").isNull())
    .select("driverId", "driverRef", "forename", "surname", "original_code", "code")
    .orderBy("driverId")
)

I preserve the original values with `withColumnRenamed(\"code\", \"original_code\")`, then normalize null-like placeholders by converting blank strings and the literal `\\\\N` value into actual nulls with `when(..., lit(None))`. After that, `regexp_replace(...)` strips punctuation, `upper(...)` standardizes the text, `substring(...)` takes the first three surname letters, and `rpad(...)` guarantees a three-character fallback for edge cases. If the assignment expects official FIA abbreviations for historical exceptions, a manual lookup table would be the best alternative route.

## Question 4

I define `Age` as the driver's completed years of life on the race date, not their age today. To answer the youngest and oldest driver question, I join race participants from `results` to the `races` and `drivers` tables, compute `Age` from `dob` and `race_date`, and then use race-level window functions to pick the youngest and oldest driver in each race.

In [0]:
driver_age_by_race = (
    results
    .select("raceId", "driverId")
    .dropDuplicates()
    .join(races.select("raceId", "year", "round", "race_name", "race_date"), on="raceId", how="left")
    .join(drivers_completed.select("driverId", "code", "forename", "surname", "dob"), on="driverId", how="left")
    .withColumn("Age", F.floor(F.months_between(F.col("race_date"), F.col("dob")) / 12))
    .withColumn("exact_age_years", F.months_between(F.col("race_date"), F.col("dob")) / 12)
)

youngest_per_race = (
    driver_age_by_race
    .withColumn(
        "row_num",
        F.row_number().over(
            Window.partitionBy("raceId").orderBy(F.col("exact_age_years").asc(), F.col("driverId").asc())
        )
    )
    .filter(F.col("row_num") == 1)
    .select(
        "raceId",
        F.col("code").alias("youngest_code"),
        F.concat_ws(" ", F.col("forename"), F.col("surname")).alias("youngest_driver"),
        F.col("Age").alias("youngest_age")
    )
)

oldest_per_race = (
    driver_age_by_race
    .withColumn(
        "row_num",
        F.row_number().over(
            Window.partitionBy("raceId").orderBy(F.col("exact_age_years").desc(), F.col("driverId").asc())
        )
    )
    .filter(F.col("row_num") == 1)
    .select(
        "raceId",
        F.col("code").alias("oldest_code"),
        F.concat_ws(" ", F.col("forename"), F.col("surname")).alias("oldest_driver"),
        F.col("Age").alias("oldest_age")
    )
)

youngest_oldest_by_race = (
    races
    .select("raceId", "year", "round", "race_name", "race_date")
    .join(youngest_per_race, on="raceId", how="left")
    .join(oldest_per_race, on="raceId", how="left")
    .orderBy("year", "round")
)

display(youngest_oldest_by_race)

The `months_between(race_date, dob) / 12` expression computes age relative to the race date, and `floor(...)` turns that value into completed years for the required `Age` column. I keep `exact_age_years` as a tie-breaker, then use `row_number().over(Window.partitionBy("raceId")...)` twice: once ordered ascending to find the youngest driver and once ordered descending to find the oldest driver. A SQL solution with two CTEs and `ROW_NUMBER()` would be a clean alternative implementation.

## Question 5

To measure podium history at each race, I start from the `results` table because it already contains the finishing order for every driver in every race. I create indicator columns for wins, second places, and third places, and then use a cumulative window by driver over race date to keep running totals of each podium type and total podiums through that race.

In [0]:
race_results_with_podium_flags = (
    results
    .filter(F.col("finish_position").isNotNull())
    .join(races.select("raceId", "year", "round", "race_name", "race_date"), on="raceId", how="left")
    .join(drivers_completed.select("driverId", "code", "forename", "surname"), on="driverId", how="left")
    .withColumn("is_win", F.when(F.col("finish_position") == 1, 1).otherwise(0))
    .withColumn("is_second", F.when(F.col("finish_position") == 2, 1).otherwise(0))
    .withColumn("is_third", F.when(F.col("finish_position") == 3, 1).otherwise(0))
)

podium_window = (
    Window.partitionBy("driverId")
    .orderBy(F.col("race_date").asc(), F.col("round").asc(), F.col("raceId").asc())
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

driver_podium_counts = (
    race_results_with_podium_flags
    .withColumn("wins_to_date", F.sum("is_win").over(podium_window))
    .withColumn("second_places_to_date", F.sum("is_second").over(podium_window))
    .withColumn("third_places_to_date", F.sum("is_third").over(podium_window))
    .withColumn(
        "podiums_to_date",
        F.col("wins_to_date") + F.col("second_places_to_date") + F.col("third_places_to_date")
    )
)

display(
    driver_podium_counts
    .select(
        "year",
        "round",
        "race_name",
        "race_date",
        "finish_position",
        "driverId",
        "code",
        "forename",
        "surname",
        "wins_to_date",
        "second_places_to_date",
        "third_places_to_date",
        "podiums_to_date"
    )
    .orderBy("year", "round", "finish_position", "driverId")
)

The three `withColumn(...)` calls that create `is_win`, `is_second`, and `is_third` turn podium finishes into numeric flags that can be summed. I then use a cumulative `Window.partitionBy("driverId").orderBy(...)` and `sum(...).over(...)` to compute the running number of wins, second places, and third places for each driver at every race, and add those pieces together to produce `podiums_to_date`. I interpret the question as totals through the end of each race; if you wanted pre-race totals instead, the same window could end at `1 PRECEDING`.

## Question 6 - My Own Exploration

My question is: which driver gained the most positions relative to the starting grid in each race? I answer it by comparing `grid_position` to `finish_position`, computing the number of places gained, and then selecting the best improvement in each race with a race-level ranking window.

In [0]:
positions_gained_by_race = (
    results
    .filter(
        (F.col("finish_position").isNotNull())
        & (F.col("grid_position").isNotNull())
        & (F.col("grid_position") > 0)
    )
    .join(races.select("raceId", "year", "round", "race_name", "race_date"), on="raceId", how="left")
    .join(drivers_completed.select("driverId", "code", "forename", "surname"), on="driverId", how="left")
    .withColumn("positions_gained", F.col("grid_position") - F.col("finish_position"))
)

biggest_mover_per_race = (
    positions_gained_by_race
    .withColumn(
        "gain_rank",
        F.row_number().over(
            Window.partitionBy("raceId").orderBy(
                F.col("positions_gained").desc(),
                F.col("finish_position").asc(),
                F.col("driverId").asc()
            )
        )
    )
    .filter(F.col("gain_rank") == 1)
)

display(
    biggest_mover_per_race
    .select(
        "year",
        "round",
        "race_name",
        "race_date",
        "driverId",
        "code",
        "forename",
        "surname",
        "grid_position",
        "finish_position",
        "positions_gained"
    )
    .orderBy("year", "round")
)

The key derived field here is `positions_gained = grid_position - finish_position`, which is positive when a driver moves forward relative to the starting grid. I rank each race with `row_number().over(Window.partitionBy("raceId").orderBy(desc("positions_gained"), asc("finish_position")))`, which returns the single biggest mover in every race while breaking ties in favor of the better finishing result. Another route would be to keep all drivers and use `dense_rank()` if you wanted to show tied biggest movers instead of just one.